In [11]:
from pydoc import describe

import pandas as pd
import numpy as np
import pyodbc
import pickle
import warnings
import time
from tqdm.notebook import tqdm
from matplotlib import pyplot as plt
tqdm.pandas()
pd.set_option('display.max_columns', 100)
pd.set_option('display.min_rows', 100)
import openpyxl
import datetime as dt
import re
import os
import copy



In [12]:
run_query = False

In [13]:
def run_sql(filename, sub_list=[], connection=None, filename_is_query=False):
    """
    Run a SQL Query by reading from a .txt file, substituting values when required.
    Input:
    filename (str): File that we want to read. Usually a .txt file.
    sub_list (list of (str,str) tuples): Substitute each instance of the first element of the tuple for the second.
                                         Example: [('{max_mob}', '6')]
    """
    if filename_is_query:
        query = filename
    else:
        with open(filename, 'r') as file:
            query = file.read()
    for sub in sub_list:
        text, var = sub
        query = query.replace(text, var)
#     print(query)
    if connection is None:
        with pyodbc.connect("DSN=Redshift_prod_new") as conn:
            warnings.filterwarnings("ignore", category=UserWarning)
            df = pd.read_sql_query(sql=query, con=conn)
            warnings.filterwarnings("default", category=UserWarning)
            return df
    else:
        warnings.filterwarnings("ignore", category=UserWarning)
        df = pd.read_sql_query(sql=query, con=connection)
        warnings.filterwarnings("default", category=UserWarning)
        return df

In [14]:
if run_query == True:
    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
                # with open("establish_temp_tables_query.txt", "r") as file:
                #     temp_table_queries = file.read()
                # # conn.execute(temp_table_queries.strip())
                # print('temptables query finished')
        df = run_sql('weekly_query', connection=conn)
        print('loss query finished')
        bl = run_sql('b2l_query', connection=conn)
        print('book to look query finished')
        tminusone = run_sql('tminusone_query', connection=conn)
        print('timinusone query finished')

        ste_df = run_sql('ste_query', connection=conn)
        print('STE query finished')
        ste_bl = run_sql('ste_b2l', connection=conn)
        print('STE b2l query finished')
        
        # Save to cache for faster reloads
        df.to_pickle('df_cache.pkl')
        bl.to_pickle('bl_cache.pkl')
        tminusone.to_pickle('tminusone_cache.pkl')
        ste_df.to_pickle('ste_df_cache.pkl')
        ste_bl.to_pickle('ste_bl_cache.pkl')
        print('Data cached to pickle files')
        
        # DEBUG: Check LOBs after data load
        print('\\n=== DEBUG: LOBs in df after load ===')
        print(df['lob'].unique())
        print(f"MCY count in df: {(df['lob'] == 'MCY').sum()}")
        print('\\n=== DEBUG: LOBs in bl after load ===')
        print(bl['lob'].unique())
        print(f"MCY count in bl: {(bl['lob'] == 'MCY').sum()}")
else:
    # Load from cache (instant!)
    df = pd.read_pickle('df_cache.pkl')
    bl = pd.read_pickle('bl_cache.pkl')
    tminusone = pd.read_pickle('tminusone_cache.pkl')
    ste_df = pd.read_pickle('ste_df_cache.pkl')
    ste_bl = pd.read_pickle('ste_bl_cache.pkl')
    print('Loaded from cache')
    
    # DEBUG: Check LOBs after load
    print('\\n=== DEBUG: LOBs in df after load ===')
    print(df['lob'].unique())
    print(f"MCY count in df: {(df['lob'] == 'MCY').sum()}")
    print('\\n=== DEBUG: LOBs in bl after load ===')
    print(bl['lob'].unique())
    print(f"MCY count in bl: {(bl['lob'] == 'MCY').sum()}")


Loaded from cache
\n=== DEBUG: LOBs in df after load ===
<ArrowStringArray>
['STG', 'KMX', 'FRN', 'MCY', 'ENT', 'AN', 'FLD', 'unassigned', '', 'ORL',
 'STE']
Length: 11, dtype: str
MCY count in df: 14835
\n=== DEBUG: LOBs in bl after load ===
<ArrowStringArray>
['KMX', 'MCY', 'STG', 'FRN', 'STE', 'AN', 'ENT', 'FLD']
Length: 8, dtype: str
MCY count in bl: 2621


In [15]:
df['application_received_dtm'] = pd.to_datetime(df['application_received_dtm'])

df['quarter'] = pd.to_datetime(df['application_received_dtm']).dt.to_period('Q')
df['month'] = pd.to_datetime(df['application_received_dtm']).dt.to_period('M')
df['week'] = pd.to_datetime(df['application_received_dtm']).dt.to_period('W-SAT') #end on saturday



bl['time'] = pd.to_datetime(bl['time'])

bl['quarter'] = pd.to_datetime(bl['time']).dt.to_period('Q')
bl['month'] = pd.to_datetime(bl['time']).dt.to_period('M')
bl['week'] = pd.to_datetime(bl['time']).dt.to_period('W-SAT') #end on saturday








# df['con_ltv_back2'] = df['con_ltv_back'] *10.0652

# count_acc_num = df['account_number'].nunique()

# countsds

#calculating
df['bbltv'] = df['con_amount_financed_back'] / df['bb_value'].replace(0, np.nan)





# df['atf_vantage'] = df['con_amount_financed_back']


# df['discount'] = np.where(df['lob'] == 'ENT', df['ent_disc'], 0)


df['luxury_flag'] = np.where( df['con_amount_financed_back'] >= 75000 , 1, 0)


# df['income_cb'] = df['income_cb'].fillna(0)

df['total_income'] = df['income_cb'].fillna(0) + df['income_pb'].fillna(0)


#assuming annual inflation of 3.5%, the average since 2020
# df['inflation_income'] = df['total_income'] / (1.035**(2025 - df['application_received_dtm'].dt.year.fillna(2025)  ))

# Set reference date as the start of 2025
# ref_date =

# Calculate number of weeks between application date and reference date
weeks_diff = ((pd.Timestamp.today() - df['application_received_dtm'].fillna(pd.Timestamp.today())).dt.days // 7)
#
#
df['inflation_income'] = df['total_income'] / ( (1.035 ** (1/52)) ** weeks_diff)
#








# df['vantage'] = pd.to_numeric(df['vantage'], errors='coerce')
df['vantage2'] = np.where( (df['vantage'] >= 300) & (df['vantage'] <= 850)  , df['vantage'], np.nan    )




#df['ent_disc']




#filtering


#folters out the actual df similar to WHERE statement1
df = df[ df['application_received_dtm'] >= '2019-10-01']
# df = df[ df['application_received_dtm'] <= '2026-01-02']

# df = df[ df['con_amount_financed_back'] <= 75000]
df = df[ (df['lob'] == 'STE') | (df['con_amount_financed_back'] <= 75000)]
df = df[ df['con_pti_back'] <= 0.6]
# df = df[ df['bbltv'] <= 4.0]
df = df[ (df['lob'] == 'MCY') | (df['lob'] == 'STE') | (df['bbltv'] <= 10.0) | (df['bb_value'].isna()) | (df['bb_value'] == 0) ]

df = df[ df['total_income'] <= 200000]



# df = df[ df['bb_value'] > 1]
# df = df[ df['total_income'] <= 20000]




# df = df[(df['application_received_dtm'] > '2025-01-01')]





#if condition is TRUE then nan, else keep original value
# df['bbltv2'] = df['bbltv'].mask( (df['bbltv'] < 0.1) | (df['bbltv'] > 0.2) , np.nan)


#
# df['discount'] = np.select( [df['lob'] == 'ENT', df['data_source_id'] == 101]
                            # , [   df['ent_disc'] , df['aca_fee'] + df['processing_fee'] ]
#                             , np.nan)

df['discount'] = np.where( df['lob'] == 'ENT', df['ent_disc'] , df['disb_acquisition_fee_amt'] )


# --- STE cleaning ---
ste_df['application_received_dtm'] = pd.to_datetime(ste_df['application_received_dtm'])
ste_df['quarter'] = ste_df['application_received_dtm'].dt.to_period('Q')
ste_df['month'] = ste_df['application_received_dtm'].dt.to_period('M')
ste_df['week'] = ste_df['application_received_dtm'].dt.to_period('W-SAT')

ste_df['vantage2'] = ste_df['vantage']

ste_weeks_diff = ((pd.Timestamp.today() - ste_df['application_received_dtm'].fillna(pd.Timestamp.today())).dt.days // 7)
ste_df['inflation_income'] = ste_df['total_income'] / ((1.035 ** (1/52)) ** ste_weeks_diff)

ste_df = ste_df[ ste_df['application_received_dtm'] >= '2019-10-01']
# ste_df = ste_df[ ste_df['con_amount_financed_back'] <= 75000]
ste_df = ste_df[ ste_df['con_pti_back'] <= 0.6]
ste_df = ste_df[ ste_df['total_income'] <= 200000]

df = pd.concat([df, ste_df], ignore_index=True)


#STE preintegration income is wrong in los deal current fact so we make it payment / PTI
ste_pre = (df['lob'] == 'STE') & (df['application_received_dtm'] < '2025-10-07')
df.loc[ste_pre, 'total_income'] = df.loc[ste_pre, 'con_payment_back_amt'] / df.loc[ste_pre, 'con_pti_back']
ste_pre_weeks = ((pd.Timestamp.today() - df.loc[ste_pre, 'application_received_dtm'].fillna(pd.Timestamp.today())).dt.days // 7)
df.loc[ste_pre, 'inflation_income'] = df.loc[ste_pre, 'total_income'] / ((1.035 ** (1/52)) ** ste_pre_weeks)

ste_bl['time'] = pd.to_datetime(ste_bl['time'])
ste_bl['quarter'] = ste_bl['time'].dt.to_period('Q')
ste_bl['month'] = ste_bl['time'].dt.to_period('M')
ste_bl['week'] = ste_bl['time'].dt.to_period('W-SAT')

bl = pd.concat([bl, ste_bl], ignore_index=True)


df['lob_genre'] = np.select( [df['lob'] == 'KMX',
                              df['lob'] == 'ENT',
                              df['lob'] == 'STE',
                              df['lob'] == 'MCY',
                              (df['lob'] == 'AN') | (df['lob'] == 'STG') | (df['lob'] == 'FLD') | (df['lob'] == 'FRN') ]
                             , ['KMX', 'ENT', 
                             'STE',
                              'MCY',
                                'NonKMXENT']
                            , np.nan
                            )



bl['lob_genre'] = np.select( [bl['lob'] == 'KMX',
                              bl['lob'] == 'ENT',
                              bl['lob'] == 'STE',
                              bl['lob'] == 'MCY',
                              (bl['lob'] == 'AN') | (bl['lob'] == 'STG') | (bl['lob'] == 'FLD') | (bl['lob'] == 'FRN') ]
                             , ['KMX', 'ENT', 'STE',
                              'MCY',
                                'NonKMXENT']
                            , np.nan)



df = df.dropna(subset=['lob_genre'])

# DEBUG: Check after lob_genre assignment and dropna
print('\\n=== DEBUG: After lob_genre assignment ===')
print('df lob_genre unique:', df['lob_genre'].unique())
print('df lob unique:', df['lob'].unique())
print(f"MCY in df after dropna: {(df['lob'] == 'MCY').sum()}")
print('\\nbl lob_genre unique:', bl['lob_genre'].unique())
print('bl lob unique:', bl['lob'].unique())
print(f"MCY in bl: {(bl['lob'] == 'MCY').sum()}")



mask = df['application_received_dtm'] >= '2023-01-01'
# print(df.loc[mask, 'bbltv'].describe())

# print(df['total_income'])



df


\n=== DEBUG: After lob_genre assignment ===
df lob_genre unique: <ArrowStringArray>
['NonKMXENT', 'KMX', 'MCY', 'ENT', 'nan', 'STE']
Length: 6, dtype: str
df lob unique: <ArrowStringArray>
['STG', 'KMX', 'FRN', 'MCY', 'ENT', 'AN', 'FLD', 'unassigned', '', 'ORL',
 'STE']
Length: 11, dtype: str
MCY in df after dropna: 14826
\nbl lob_genre unique: <ArrowStringArray>
['KMX', 'MCY', 'NonKMXENT', 'STE', 'ENT']
Length: 5, dtype: str
bl lob unique: <ArrowStringArray>
['KMX', 'MCY', 'STG', 'FRN', 'STE', 'AN', 'ENT', 'FLD']
Length: 8, dtype: str
MCY in bl: 2621


,account_number,con_payment_back_amt,con_ltv_back,con_amount_financed_back,disb_acquisition_fee_amt,ent_disc,con_apr,application_received_dtm,con_pti_back,con_cash_down_amt,purchase_odometer,purchase_year,book_date,risk_model_name,risk_model_version,con_risk_model_score,veh_age,sfs_fico,lob,income_pb,income_cb,adj_fico,vantage,bb_value,data_source_id,aca_fee,aca_usury_fee_amt,processing_fee,dealer_part_amt,stellantis_make_flag,purchase_type,received_date,watch_list_fee_amt,quarter,month,week,bbltv,luxury_flag,total_income,inflation_income,vantage2,discount,pricing_discount,management_fee,dealer_flats,watch_list_fee,discount_percent,total_income_dec,lob_genre
0,9.012381e+10,809.85,1.5314,33041.00,2994.3808,2544.3808,0.2060,2019-12-28 12:02:36.247,0.0734,1500.00,36200.0,2016.0,2020-01-06,PANDA,2.0,148.0,4.333333,NaN,STG,11039.87,0.0,536.0,536.0,19675.0,17.0,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,2019Q4,2019-12,2019-12-22/2019-12-28,1.679339,0.0,11039.87,8868.759677,536.0,2994.3808,NaN,NaN,NaN,NaN,NaN,NaN,NonKMXENT
1,9.012381e+10,368.57,1.0541,12647.00,1000.0000,750.0000,0.2800,2020-01-03 19:27:02.413,0.2074,1500.00,24398.0,2017.0,2020-01-07,MOUNTAIN,2.0,120.0,3.333333,NaN,KMX,1776.67,0.0,446.0,446.0,8050.0,17.0,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,2020Q1,2020-01,2019-12-29/2020-01-04,1.571056,0.0,1776.67,1428.213095,446.0,1000.0000,NaN,NaN,NaN,NaN,NaN,NaN,KMX
2,9.012381e+10,621.25,1.3628,21802.46,1000.0000,550.0000,0.2700,2020-01-03 19:36:49.357,0.0954,500.00,15021.0,2017.0,2020-01-06,MOUNTAIN,1.0,139.0,3.333333,NaN,KMX,6511.26,0.0,537.0,537.0,12850.0,17.0,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,2020Q1,2020-01,2019-12-29/2020-01-04,1.696689,0.0,6511.26,5234.211643,537.0,1000.0000,NaN,NaN,NaN,NaN,NaN,NaN,KMX
3,9.012382e+10,672.36,1.4947,21673.80,2121.7388,1871.7388,0.2500,2020-01-21 18:43:00.017,0.1939,1800.00,137967.0,2012.0,2020-01-29,PANDA,2.0,116.0,8.333333,NaN,FRN,3466.67,0.0,518.0,518.0,13100.0,17.0,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,2020Q1,2020-01,2020-01-19/2020-01-25,1.654489,0.0,3466.67,2790.444395,518.0,2121.7388,NaN,NaN,NaN,NaN,NaN,NaN,NonKMXENT
4,9.012382e+10,426.14,1.2726,15268.57,1000.0000,550.0000,0.2438,2020-01-22 21:01:43.110,0.1567,300.00,101130.0,2014.0,2020-01-24,MOUNTAIN,2.0,142.0,6.333333,NaN,KMX,2719.09,0.0,0.0,NaN,NaN,17.0,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,2020Q1,2020-01,2020-01-19/2020-01-25,NaN,0.0,2719.09,2188.691006,NaN,1000.0000,NaN,NaN,NaN,NaN,NaN,NaN,KMX
5,9.012382e+10,684.71,1.0216,23495.00,1000.0000,550.0000,0.2800,2020-01-23 21:22:35.843,0.1141,2199.89,25926.0,2016.0,2020-01-24,MOUNTAIN,2.0,138.0,4.333333,NaN,KMX,6000.02,0.0,566.0,566.0,18675.0,17.0,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,2020Q1,2020-01,2020-01-19/2020-01-25,1.258099,0.0,6000.02,4829.626753,566.0,1000.0000,NaN,NaN,NaN,NaN,NaN,NaN,KMX
6,9.012383e+10,317.88,1.5076,10907.57,1653.8911,1203.8911,0.2800,2020-01-25 12:20:34.790,0.0636,NaN,12619.0,2008.0,2020-02-04,MCYMODEL,NaN,140.0,12.416667,NaN,MCY,5000.00,0.0,476.0,476.0,NaN,17.0,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,2020Q1,2020-01,2020-01-19/2020-01-25,NaN,0.0,5000.00,4027.339014,476.0,1653.8911,NaN,NaN,NaN,NaN,NaN,NaN,MCY
7,9.012383e+10,386.40,0.8084,16007.01,1065.7593,615.7593,0.2000,2020-01-25 12:48:32.503,0.0577,10000.00,34556.0,2017.0,2020-01-30,PANDA,2.0,161.0,3.333333,NaN,STG,2700.00,4000.0,0.0,NaN,18300.0,17.0,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,2020Q1,2020-01,2020-01-19/2020-01-25,0.874700,0.0,6700.00,5396.634279,NaN,1065.7593,NaN,NaN,NaN,NaN,NaN,NaN,NonKMXENT
8,9.012382e+10,557.73,0.9570,19137.91,1000.0000,750.0000,0.2800,2020-01-25 22:03:53.707,0.0705,5400.00,126797.0,2011.0,2020-01-27,MOUNTAIN,2.0,117.0,9.333333,NaN,KMX,7916.00,0.0,407.0,407.0,NaN,17.0,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,2020Q1,2020-01,2020-01-19/2020-01-25,NaN,0.0,7916.00,6376.083127,407.0,1000.0000,NaN,NaN,NaN,NaN,NaN,NaN,KMX
9,9.012383e+10,525.85,1.0025,18043.89,1000.0000,750.0000,0.2800,2020-01-31 12:58:03.693,0.1683,1000.00,58536.0,2017.0,2020-02-04,MOUNTAIN,2.0,126.0,3.416667,NaN,KMX,3125.01,0.0,497.0,497.0,13200.0,17.0,NaN,NaN,NaN,Na

In [16]:
#aggregating

granularity = ['quarter','month' , 'week'][2]
lob_granularity = ['lob_genre', 'lob'][0]



def generate_report(df, bl, granularity, lob_granularity):
    # DEBUG: Check what comes into generate_report
    print(f'\\n=== DEBUG: Inside generate_report, lob_granularity={lob_granularity} ===')
    print(f'df[lob_granularity].unique(): {df[lob_granularity].unique()}')
    print(f'bl[lob_granularity].unique(): {bl[lob_granularity].unique()}')
    if lob_granularity == 'lob':
        print(f"MCY in df['lob']: {(df['lob'] == 'MCY').sum()}")
        print(f"MCY in bl['lob']: {(bl['lob'] == 'MCY').sum()}")
    # Find two Saturdays ago
    today = pd.Timestamp.today().normalize()
    days_since_saturday = (today.weekday() - 5) % 7
    last_saturday = today - pd.Timedelta(days=days_since_saturday)
    two_saturdays_ago = last_saturday - pd.Timedelta(days=7)

    # Get periods
    week_ref = pd.Period(two_saturdays_ago, freq='W-SAT')
    month_ref = pd.Period(two_saturdays_ago, freq='M')

    if granularity == 'quarter':
        min_quarter = pd.Period('2020Q1')
        df = df[df['quarter'] >= min_quarter]
    if granularity == 'month':
        last_5_months = month_ref - 4
        df = df[(df['month'] >= last_5_months) & (df['month'] <= month_ref)]
    if granularity == 'week':
        if lob_granularity == 'lob':
            last_n_weeks = 4  # only 5 weeks for tzero
        else:
            last_n_weeks = 5   # 6 weeks total (current + 5 previous)
        last_weeks = week_ref - last_n_weeks
        df  = df[(df['week'] >= last_weeks) & (df['week'] <= week_ref)]


        # last_6_weeks = week_ref - 5
        # df = df[(df['week'] >= last_6_weeks) & (df['week'] <= week_ref)]




    quarterly_df = df.groupby([lob_granularity, granularity]).apply(
        lambda g: pd.Series({



            # 'vantage' : (g['vantage2'] * g['con_amount_financed_back']).sum() / g['con_amount_financed_back'].sum(),

            'vantage': (g.loc[g['vantage2'].notnull(), 'vantage2'] * g.loc[g['vantage2'].notnull(), 'con_amount_financed_back']).sum() / g.loc[g['vantage2'].notnull(), 'con_amount_financed_back'].sum(),


            # 'vantage3' : g['vantage2'].sum(skipna=True),
            # 'vantage4' : g['vantage2'].mean(),

            # 'vantage5': np.average(g['vantage2'], weights=g['con_amount_financed_back']),
                         # * g['con_amount_financed_back']).sum() / g['con_amount_financed_back'].sum(),

            'model score weighted': (g['con_risk_model_score'] * g['con_amount_financed_back']).sum() / g['con_amount_financed_back'].sum(),

            'cash down avg': g['con_cash_down_amt'].mean(),
            'cash down wtd': (g['con_cash_down_amt'] * g['con_amount_financed_back']).sum() / g['con_amount_financed_back'].sum(),

            'discount avg': g['discount'].mean(),
            'discount pct': (g['discount']).sum() / g['con_amount_financed_back'].sum(),

            'amount financed avg': g['con_amount_financed_back'].mean(),

            'apr avg': g['con_apr'].mean(),
            'apr wtd': (g['con_apr'] * g['con_amount_financed_back']).sum() / g['con_amount_financed_back'].sum(),

            'total income avg': g['total_income'].mean(),
            'inflation_adjusted_income avg': g['inflation_income'].mean(),

            'payment avg': g['con_payment_back_amt'].mean(),

            'pti wtd' : (g['con_pti_back']* g['con_amount_financed_back']).sum() / g['con_amount_financed_back'].sum(),

            'contracts num': g['account_number'].nunique(),
            'duplicates': g['account_number'].duplicated().sum(),

            # 'contracts 2': len(g['account_number'].unique()),
        # .count(),
        #     'contracts 3': g['account_number'].drop_duplicates().count(),
        'all contracts': g['account_number'].count(),


    # VEHICLE METRICS
            'Blackbook Value avg': (g['bb_value'].mean()), #* g['con_amount_financed_back']).sum() / g['con_amount_financed_back'].sum() ,
            'bbltv avg': g['bbltv'].mean(),
            # 'bbltv weighted': (g['bbltv'] * g['con_amount_financed_back']).sum() / g['con_amount_financed_back'].sum(),
            # 'bbltv 2weighted': (g.loc[g['bbltv'] > 1, 'bbltv'] * g.loc[g['bbltv'] > 1, 'con_amount_financed_back']).sum() / g.loc[g['bbltv'] > 1, 'con_amount_financed_back'].sum(),
            'bbltv 2weighted': (g.loc[g['bbltv'].notnull(), 'bbltv'] * g.loc[g['bbltv'].notnull(), 'con_amount_financed_back']).sum() / g.loc[g['bbltv'].notnull(), 'con_amount_financed_back'].sum(),

            'mileage avg': g['purchase_odometer'].mean(),
            'vehicle age avg':g['veh_age'].mean(),


        })
    ).reset_index()

    # DEBUG: Check quarterly_df after groupby
    print(f'\\n=== DEBUG: quarterly_df LOBs after groupby ===')
    print(f"quarterly_df[lob_granularity].unique(): {quarterly_df[lob_granularity].unique()}")
    print(f"MCY in quarterly_df: {(quarterly_df[lob_granularity] == 'MCY').sum()}")




    b2l_df = bl.groupby([lob_granularity, granularity]).apply(
        lambda g: pd.Series({

            # 'b2l avg': g['b2l'].mean(),
            'b2l': g['cons'].sum() / g['apps'].sum(),
            'applications': g['apps'].sum(),

        })
    ).reset_index()


    merged_df = pd.merge(quarterly_df, b2l_df, on=[lob_granularity, granularity], how='left')

    # DEBUG: Check b2l_df and merged_df
    print(f'\\n=== DEBUG: b2l_df LOBs ===')
    print(f"b2l_df[lob_granularity].unique(): {b2l_df[lob_granularity].unique()}")
    print(f"MCY in b2l_df: {(b2l_df[lob_granularity] == 'MCY').sum()}")
    print(f'\\n=== DEBUG: merged_df LOBs ===')
    print(f"merged_df[lob_granularity].unique(): {merged_df[lob_granularity].unique()}")
    print(f"MCY in merged_df: {(merged_df[lob_granularity] == 'MCY').sum()}")



    # Melt to long format: one row per lob_genre, quarter, measure, value
    melted = merged_df.melt(id_vars=[lob_granularity, granularity], var_name='measure', value_name='value')

    # Pivot so quarters are columns, measures are rows, lob_genre repeats for each measure
    pivoted = melted.pivot_table(index=[lob_granularity, 'measure'], columns=granularity, values='value')

    # Optional: reset index for a clean DataFrame
    pivoted = pivoted.reset_index()

    # List of measures in your desired order
    measure_order = [
        
        'model score weighted',
        'discount avg', 'discount pct', 
        'apr avg', 'apr wtd',
        'applications',
        'contracts num',
        # 'b2l'
        'Blackbook Value avg',
        'bbltv 2weighted',
        'mileage avg',
        'vehicle age avg',
    ]

    # After melting
    melted['measure'] = pd.Categorical(melted['measure'], categories=measure_order, ordered=True)

    # Now pivot as before
    pivoted = melted.pivot_table(index=[lob_granularity, 'measure'], columns=granularity, values='value')


    pivoted.reset_index()

    return pivoted
    # pivoted















In [17]:


# Create POS aggregate LOB (sum of all individual LOBs)
pos_lobs = ['KMX', 'ENT', 'FLD', 'FRN', 'AN', 'STG', 'MCY']

# Create POS copies for df
df_pos = df[df['lob'].isin(pos_lobs)].copy()
df_pos['lob'] = 'POS'
df_with_pos = pd.concat([df, df_pos], ignore_index=True)

# Create POS copies for bl
bl_pos = bl[bl['lob'].isin(pos_lobs)].copy()
bl_pos['lob'] = 'POS'
bl_with_pos = pd.concat([bl, bl_pos], ignore_index=True)

print(f"POS records in df: {(df_with_pos['lob'] == 'POS').sum()}")
print(f"POS records in bl: {(bl_with_pos['lob'] == 'POS').sum()}")

# generate_report(df, bl, 'quarter', 'lob_genre')
# quarterly_df = generate_report(df, bl, 'quarter', 'lob_genre')

# monthly_df = generate_report(df, bl, 'month', 'lob_genre')

# weekly_df = generate_report(df, bl, 'week', 'lob_genre')

# Use df_with_pos and bl_with_pos to include POS aggregate
tzero_df = generate_report(df_with_pos, bl_with_pos, 'week', 'lob')


# quarterly_lob_breakout = generate_report(df, bl, 'quarter', 'lob')




tzero_df



POS records in df: 797421
POS records in bl: 18131
\n=== DEBUG: Inside generate_report, lob_granularity=lob ===
df[lob_granularity].unique(): <ArrowStringArray>
[       'STG',        'KMX',        'FRN',        'MCY',        'ENT',
         'AN',        'FLD', 'unassigned',           '',        'ORL',
        'STE',        'POS']
Length: 12, dtype: str
bl[lob_granularity].unique(): <ArrowStringArray>
['KMX', 'MCY', 'STG', 'FRN', 'STE', 'AN', 'ENT', 'FLD', 'POS']
Length: 9, dtype: str
MCY in df['lob']: 14826
MCY in bl['lob']: 2621


C:\Users\ahmed.ali\AppData\Local\Temp\ipykernel_32948\427858714.py:97: RuntimeWarning: invalid value encountered in scalar divide
  'bbltv 2weighted': (g.loc[g['bbltv'].notnull(), 'bbltv'] * g.loc[g['bbltv'].notnull(), 'con_amount_financed_back']).sum() / g.loc[g['bbltv'].notnull(), 'con_amount_financed_back'].sum(),


\n=== DEBUG: quarterly_df LOBs after groupby ===
quarterly_df[lob_granularity].unique(): <ArrowStringArray>
['AN', 'ENT', 'FLD', 'FRN', 'KMX', 'MCY', 'ORL', 'POS', 'STE', 'STG']
Length: 10, dtype: str
MCY in quarterly_df: 5


C:\Users\ahmed.ali\AppData\Local\Temp\ipykernel_32948\427858714.py:118: RuntimeWarning: divide by zero encountered in scalar divide
  'b2l': g['cons'].sum() / g['apps'].sum(),


\n=== DEBUG: b2l_df LOBs ===
b2l_df[lob_granularity].unique(): <ArrowStringArray>
['AN', 'ENT', 'FLD', 'FRN', 'KMX', 'MCY', 'POS', 'STE', 'STG']
Length: 9, dtype: str
MCY in b2l_df: 384
\n=== DEBUG: merged_df LOBs ===
merged_df[lob_granularity].unique(): <ArrowStringArray>
['AN', 'ENT', 'FLD', 'FRN', 'KMX', 'MCY', 'ORL', 'POS', 'STE', 'STG']
Length: 10, dtype: str
MCY in merged_df: 5


C:\Users\ahmed.ali\AppData\Local\Temp\ipykernel_32948\427858714.py:162: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  melted['measure'] = pd.Categorical(melted['measure'], categories=measure_order, ordered=True)


week                      2026-03-22/2026-03-28  2026-03-29/2026-04-04  \
lob measure                                                              
AN  model score weighted             140.565137             140.949519   
    discount avg                    1170.178540            1159.338552   
    discount pct                       0.052535               0.050184   
    apr avg                            0.245362               0.248471   
    apr wtd                            0.238245               0.242606   
    applications                    3270.000000            3057.000000   
    contracts num                    213.000000             194.000000   
    Blackbook Value avg            15371.192488           15501.871134   
    bbltv 2weighted                    1.584679               1.603810   
    mileage avg                    83314.483568           75807.180412   
    vehicle age avg                    7.110329               7.045103   
ENT model score weighted             140.080841             140.640051   
    discount avg                     899.962742             884.645315   
    discount pct                       0.037829               0.037295   
    apr avg                            0.237284               0.237123   
    apr wtd                            0.236794               0.237169   
    applications                   10092.000000            9443.000000   
    contracts num                    310.000000             286.000000   
    Blackbook Value avg            18303.951613           18306.993007   
    bbltv 2weighted                    1.322360               1.322901   
    mileage avg                    49756.309677           50731.786713   
    vehicle age avg                    2.973925               2.930361   
FLD model score weighted             141.377108             141.177767   
    discount avg                     483.809839             479.921174   
    discount pct                       0.021782               0.021486   
    apr avg                            0.239271               0.237526   
    apr wtd                            0.238554               0.235722   
    applications                    5048.000000            4445.000000   
    contracts num                    248.000000             230.000000   
    Blackbook Value avg            15994.104839           15803.917391   
...                                         ...                    ...   
POS apr avg                            0.248093               0.247425   
    apr wtd                            0.244862               0.243861   
    applications                  115108.000000          106930.000000   
    contracts num                   3708.000000            3487.000000   
    Blackbook Value avg            17205.079508           17384.447575   
    bbltv 2weighted                    1.514390               1.521105   
    mileage avg                    60956.719256           61354.460855   
    vehicle age avg                    5.767732               5.807858   
STE model score weighted             135.042149             134.999902   
    discount avg                     452.338344             500.294092   
    discount pct                       0.014482               0.016034   
    apr avg                            0.228458               0.231564   
    apr wtd                            0.224452               0.225706   
    applications                   17129.000000           15872.000000   
    contracts num                    748.000000             650.000000   
    Blackbook Value avg            22112.850467           22168.192308   
    bbltv 2weighted                    1.444995               1.440015   
    mileage avg                    52381.675567           53639.881538   
    vehicle age avg                    4.383850               4.500484   
STG model score weighted             140.923489             141.108738   
    discount avg                    1232.902124            1146.565365   
    

In [18]:
tminusone



,roa_dealer_group,rowid,lookup,notes,datetime_inserted,effective_date,halfbaked_date,mroa,orig_yield,cnl_effect,var_cost,cost_of_debt,run_rate_nb,model_score,model_name,wal,wal_cfc,wal_adj_factor,apr,apr_realization_factor,discount,fee_income,pmt_proc_income,addl_income,cnl,gross_loss,recovery,cnl_adj_factor,gross_loss_cfc,gross_loss_adj_factor,recovery_cfc,recovery_adj_factor,debt_int_rate,securitization_cost,debt_adv_rate,orig_cost,serv_cost,addl_cost,days_till_half_baked,sort,latest_row_flag,loss_meeting_flag,pricing_change_flag,is_current_flag,elasticity,nb_ratio,version_start_datetime,version_end_datetime,ticket_no,addl_cost_of_debt,tier,fixed_cost,subdebt_expense,vroa,ragu_score,baseline_ltv,ltv,baseline_apr,ragu_gross_loss_adjustment,ragu_recovery_adjustment,ragu_ltv_adjustment,ragu_market_expectation_adjustment,ragu_apr_adjustment
0,AN,404,AN404,Cost of Funds Update (5.7%),2026-05-05,2026-05-04,None,0.070162,0.265065,0.118825,0.027206,0.048872,200.0,139.6,Franchise3.0,2.12,2.217180,0.956170,0.250766,0.922800,0.052700,0.003,0.0041,0.0017,0.251909,0.578196,0.326287,1.319580,0.438167,1.0,0.247266,1.0,0.057,0.0026,0.82,0.031601,0.0123,0,16,2,1,0,0,1,0.0100,0.9935,2026-05-05 18:30:59,9999-01-01,None,0.0,None,0.005,0.0027,0.062462,147.3,1.94,1.58,0.250,3.6,0.50,3.9,0.0,-0.3
1,ENT,291,ENT291,Cost of Funds Update (5.7%),2026-05-05,2026-05-04,None,0.060504,0.249941,0.113351,0.027214,0.048872,304.0,139.6,Franchise3.0,2.39,2.304859,1.036940,0.240285,0.925900,0.044600,0.003,0.0041,0.0017,0.270909,0.563766,0.292857,1.140854,0.494162,1.0,0.256700,1.0,0.057,0.0026,0.82,0.035645,0.0123,0,13,4,1,0,0,1,0.0100,0.9935,2026-05-05 18:31:56,9999-01-01,None,0.0,None,0.005,0.0027,0.052804,145.0,1.45,1.33,0.235,1.2,3.00,1.5,0.0,-0.3
2,FLD,292,FLD292,Cost of Funds Update (5.7%),2026-05-05,2026-05-04,None,0.055630,0.236927,0.104822,0.027602,0.048872,188.1,139.9,Franchise3.0,2.30,2.228464,1.032101,0.236701,0.933000,0.016755,0.003,0.0041,0.0017,0.241091,0.697602,0.456511,1.854228,0.376222,1.0,0.246200,1.0,0.057,0.0026,0.82,0.035195,0.0123,0,13,5,1,0,0,1,0.0100,0.9935,2026-05-05 18:32:34,9999-01-01,None,0.0,None,0.005,0.0027,0.047930,144.0,1.45,1.45,0.235,2.5,1.70,0.0,0.0,-0.1
3,FRN,416,FRN416,Cost of Funds Update (5.7%),2026-05-05,2026-05-04,None,0.057350,0.274817,0.141318,0.027277,0.048872,598.5,142.3,Franchise3.0,2.00,2.178676,0.917989,0.256453,0.905300,0.067700,0.003,0.0041,0.0017,0.282636,0.612515,0.329879,1.317600,0.464872,1.0,0.250364,1.0,0.057,0.0026,0.82,0.029954,0.0123,0,19,6,1,0,0,1,0.0100,0.9930,2026-05-05 18:33:52,9999-01-01,None,0.0,None,0.005,0.0027,0.049650,151.0,1.94,1.44,0.250,3.4,0.00,5.9,0.0,-0.6
4,KMX,433,KMX433,Cost of Funds Update (5.7%),2026-05-05,2026-05-04,None,0.063434,0.248586,0.119404,0.016876,0.048872,1764.0,142.6,Mountain3,2.38,2.417356,0.984547,0.244793,0.920300,0.032377,0.003,0.0041,0.0026,0.284182,0.652518,0.368337,1.307707,0.498979,1.0,0.281666,1.0,0.057,0.0026,0.82,0.011368,0.0121,0,4,1,1,0,0,1,0.0065,0.9400,2026-05-05 18:29:15,9999-01-01,None,0.0,None,0.005,0.0027,0.055734,144.0,1.59,1.57,0.250,0.2,0.47,0.2,0.0,0.5
5,MCY,345,MCY345,Cost of Funds Update (5.7%),2026-05-05,2026-05-04,None,0.053134,0.301087,0.171804,0.027277,0.048872,30.0,148.7,MCY2.0,1.97,2.243515,0.878086,0.252000,0.890727,0.133613,0.003,0.0041,0.0017,0.338454,0.585907,0.247453,1.453143,0.403200,1.0,0.170288,1.0,0.057,0.0026,0.82,0.029504,0.0123,0,12,7,1,0,0,1,0.0117,0.9915,2026-05-05 18:33:21,9999-01-01,None,0.0,None,0.005,0.0027,0.045434,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,SFS,41,SFS41,Cost of Funds Update (5.7%),2026-05-05,2026-05-04,None,0.045233,0.226153,0.117048,0.015000,0.048872,710.0,0.0,,2.27,2.270000,1.000000,0.228000,0.905000,0.025000,0.003,0.0041,0.0017,0.265700,0.468001,0.202301,1.025869,0.456200,1.0,0.197200,1.0,0.057,0.0026,0.82,0.005675,0.0125,0,2,10,1,0,0,1,0.0100,1.0000,2026-05-05 18:34:12,9999-01-01,None,0.0,None,0.005,0.0027,0.037533,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,STG,423,STG423,Cost of Funds Update (5.7%),2026-05-05,20

In [19]:
# pivoted
#

In [20]:


# import pandas as pd

def format_time_columns(df):
    new_cols = []
    for col in df.columns:
        if isinstance(col, pd.Period):
            if col.freqstr == 'M':
                new_cols.append(col.start_time.strftime('%b %y'))
            elif col.freqstr.startswith('W'):
                new_cols.append(col.start_time.strftime('%d-%b-%y'))
            elif col.freqstr.startswith('Q'):
                new_cols.append(f"{col.year} Q{col.quarter}")
            else:
                new_cols.append(col)
        else:
            new_cols.append(col)
    df = df.copy()
    df.columns = new_cols
    return df

# Usage:
tzero_fmt = format_time_columns(tzero_df)
# monthly_fmt = format_time_columns(monthly_df)
# weekly_fmt = format_time_columns(weekly_df)
# lob_breakout_fmt = format_time_columns(lob_breakout_df)
# quarterly_lob_breakout_fmt = format_time_columns(quarterly_lob_breakout)













with pd.ExcelWriter('tzero_results.xlsx', engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    tzero_fmt.to_excel(writer, sheet_name='tzero_results', index=True)
    tminusone.to_excel(writer, sheet_name='tminusone', index=True)
    # monthly_fmt.to_excel(writer, sheet_name='lobgenre_month', index=True)
    # weekly_fmt.to_excel(writer, sheet_name='lobgenre_week', index=True)
    # lob_breakout_fmt.to_excel(writer, sheet_name='lob_week', index=True)
    # quarterly_lob_breakout_fmt.to_excel(writer, sheet_name='alllob_quarter', index=True)

